# WAXAL ASR Submission — Gemma 3n + LoRA

Fine-tune Gemma 3n for ASR on Lingala, Shona, and Luganda.

**Hardware:** T4 GPU (Colab free) or A100 (Colab Pro)

**Rules:** Open-source only, seed=42, no AutoML, max 5 submissions/day

In [ ]:
# Install dependencies
!pip install -q transformers>=4.52.0 datasets peft trl jiwer timm librosa soundfile accelerate bitsandbytes huggingface_hub torchao
!apt-get -qq install ffmpeg
print('Dependencies installed')

In [ ]:
# Configuration
import dataclasses
from typing import Final
import os, random, numpy as np, torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_ID = 'google/gemma-3n-E2B-it'
DATASET_ID = 'google/WaxalNLP'
LANGUAGES = ['lin', 'sna', 'lug']
SAMPLE_RATE = 16_000
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.0
LORA_TARGET_MODULES = ['v_proj', 'o_proj']
MAX_STEPS = 500
BATCH_SIZE = 2
GRADIENT_ACCUMULATION = 8
LEARNING_RATE = 1e-3
SYSTEM_MESSAGE = 'You are an assistant that transcribes speech accurately.'
USER_MESSAGE = 'Please transcribe this audio.'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 1. Hugging Face Login

**Required.** Get token at https://huggingface.co/settings/tokens
Accept license at https://huggingface.co/google/gemma-3n-E2B-it

In [ ]:
from huggingface_hub import login
login()

## 2. Load Dataset (all 3 languages)

In [ ]:
import datasets
import librosa
import numpy as np
from collections.abc import Mapping, Sequence
from typing import Any

def format_for_chat(example, system_message=SYSTEM_MESSAGE, user_message=USER_MESSAGE):
    audio = example['audio']
    audio_array = audio['array']
    return {
        **example,
        'messages': [
            {'role': 'system', 'content': [{'type': 'text', 'text': system_message}]},
            {'role': 'user', 'content': [{'type': 'audio', 'audio': audio_array}, {'type': 'text', 'text': user_message}]},
            {'role': 'assistant', 'content': [{'type': 'text', 'text': str(example['transcription'])}]},
        ],
    }

def format_batch(batch):
    num = len(batch['transcription'])
    examples = [{k: batch[k][i] for k in batch} for i in range(num)]
    formatted = [format_for_chat(ex) for ex in examples]
    return {k: [ex[k] for ex in formatted] for k in formatted[0]}

# Load all 3 languages
all_train, all_val = [], []
for lang in LANGUAGES:
    print(f'Loading {lang}...')
    train = datasets.load_dataset(DATASET_ID, name=f'{lang}_asr', split='train', streaming=True)
    val = datasets.load_dataset(DATASET_ID, name=f'{lang}_asr', split='validation', streaming=True)
    train = train.cast_column('audio', datasets.Audio(sampling_rate=SAMPLE_RATE))
    val = val.cast_column('audio', datasets.Audio(sampling_rate=SAMPLE_RATE))
    # Small subset for faster first run (remove for full)
    train = train.take(500)
    val = val.take(100)
    all_train.append(train)
    all_val.append(val)

from datasets import interleave_datasets
train_ds = interleave_datasets(all_train)
val_ds = interleave_datasets(all_val)

train_ds = train_ds.map(format_batch, batched=True, batch_size=32)
val_ds = val_ds.map(format_batch, batched=True, batch_size=32)

print(f'Train samples: {len(list(train_ds))}, Val: {len(list(val_ds))}')

## 3. Load Model + Apply LoRA

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

# Quantization for T4
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    quantization_config=quant_config,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

# Apply LoRA
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4. Train with LoRA

In [ ]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir='./gemma-waxal',
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    max_steps=MAX_STEPS,
    max_seq_length=64,
    logging_steps=10,
    save_steps=100,
    eval_steps=100,
    evaluation_strategy='steps',
    report_to=['none'],
    fp16=True,
    seed=SEED,
    data_seed=SEED,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=processor,
)

trainer.train()
trainer.save_model('./gemma-waxal-best')

## 5. Evaluate & Submit

In [ ]:
from jiwer import wer, cer
import pandas as pd

model.eval()

# Generate predictions on test set
test = datasets.load_dataset(DATASET_ID, 'lin_asr', split='test', streaming=True)
test = test.cast_column('audio', datasets.Audio(sampling_rate=SAMPLE_RATE))

predictions = []
for i, example in enumerate(test.take(50)):
    audio = example['audio']['array']
    inputs = processor(text=[USER_MESSAGE], audio=[audio], return_tensors='pt', sampling_rate=SAMPLE_RATE)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=50)
    pred = processor.decode(outputs[0], skip_special_tokens=True)
    predictions.append({'id': example['id'], 'transcription': pred})

df = pd.DataFrame(predictions)
df.to_csv('../submissions/gemma_submission.csv', index=False)
print(df.head())
print(f'Saved: {len(df)} predictions')